In [1]:
# General notebook settings
import logging
import warnings

import pypsa

warnings.filterwarnings("error", category=DeprecationWarning)
# pandas<3.0.3 sets the `locs` attribute deprecated in matplotlib>=3.11
warnings.filterwarnings("ignore", message="The locs attribute was deprecated")
logging.getLogger("gurobipy").propagate = False
pypsa.options.params.optimize.log_to_console = False

# Unit Commitment

This tutorial runs through examples of unit commitment for generators at a single bus. Examples of minimum part-load, minimum up time, minimum down time, start up costs, shut down costs and ramp rate restrictions are shown, as well as how to set up a rolling horizon optimization.

To enable unit commitment on a component (`Link` or `Generator`), set its attribute `committable=True`.

In [2]:
import pandas as pd

import pypsa

## Minimum Part Load

In the final snapshot, the load goes below the part-load limit of the coal generator (30%), forcing gas to commit.

In [3]:
nu = pypsa.Network(snapshots=range(4))

nu.add("Bus", "bus")

nu.add(
    "Generator",
    "coal",
    bus="bus",
    committable=True,
    p_min_pu=0.3,
    marginal_cost=20,
    p_nom=10_000,
)

nu.add(
    "Generator",
    "gas",
    bus="bus",
    committable=True,
    marginal_cost=70,
    p_min_pu=0.1,
    p_nom=1_000,
)

nu.add("Load", "load", bus="bus", p_set=[4_000, 6_000, 5_000, 800])

/tmp/ipykernel_8856/181200406.py:1: PyPSA2Warning: pandas reads strings as `str` dtype since 3.0. PyPSA converts them back to object dtype on import, but keeps `str` from PyPSA 2.0 on. Set `pypsa.options.future.string_dtype` to True to opt in now or to False to keep the current behaviour.
  nu = pypsa.Network(snapshots=range(4))


In [4]:
nu.optimize()

/tmp/ipykernel_8856/675807165.py:1: PyPSA2Warning: From PyPSA 2.0 on, the objective leaves out the capital costs of existing capacity on extendable assets, which improves LP conditioning. Pass `include_objective_constant` to `n.optimize` to choose per call. Set `pypsa.options.future.exclude_objective_constant` to True to opt in now or to False to keep the current behaviour.
  nu.optimize()
Index(['bus'], dtype='object', name='name')


INFO:linopy.model: Solve problem using Gurobi solver


INFO:linopy.model:Solver options:
 - log_to_console: False


INFO:linopy.io: Writing time: 0.04s


Set parameter WLSAccessID


Set parameter WLSSecret


... (5 outputs left out)

Set parameter LogToConsole to value 0


INFO:linopy.constants: Optimization successful: 
Status: ok
Termination condition: optimal
Solution: 32 primals, 0 duals
Objective: 3.56e+05
Solver: gurobi
Runtime: 0.00s
MIP gap: 0.00e+00
Dual bound: 3.56e+05
Solver model: available
Solver message: 2



INFO:pypsa.optimization.optimize:No shadow prices were assigned to the network.


('ok', 'optimal')

In [5]:
nu.generators_t.status

name,coal,gas
snapshot,,
0,1.0,0.0
1,1.0,0.0
2,1.0,0.0
3,0.0,1.0


In [6]:
nu.generators_t.p

name,coal,gas
snapshot,,
0,4000.0,0.0
1,6000.0,0.0
2,5000.0,0.0
3,0.0,800.0


## Minimum Up Time

Gas has a minimum up time, forcing it to be online longer than otherwise necessary, which incurs a standby cost for status up without generation.

In [7]:
nu = pypsa.Network(snapshots=range(4))

nu.add("Bus", "bus")

nu.add(
    "Generator",
    "coal",
    bus="bus",
    committable=True,
    p_min_pu=0.3,
    marginal_cost=20,
    p_nom=10000,
)

nu.add(
    "Generator",
    "gas",
    bus="bus",
    committable=True,
    stand_by_cost=50,
    marginal_cost=70,
    p_min_pu=0.1,
    up_time_before=0,
    min_up_time=3,
    p_nom=1_000,
)

nu.add("Load", "load", bus="bus", p_set=[4_000, 800, 5_000, 3_000])

In [8]:
nu.optimize()

Index(['bus'], dtype='object', name='name')


INFO:linopy.model: Solve problem using Gurobi solver


INFO:linopy.model:Solver options:
 - log_to_console: False


INFO:linopy.io: Writing time: 0.03s


Set parameter WLSAccessID


Set parameter WLSSecret


... (5 outputs left out)

Set parameter LogToConsole to value 0


INFO:linopy.constants: Optimization successful: 
Status: ok
Termination condition: optimal
Solution: 32 primals, 0 duals
Objective: 3.06e+05
Solver: gurobi
Runtime: 0.00s
MIP gap: 0.00e+00
Dual bound: 3.06e+05
Solver model: available
Solver message: 2



INFO:pypsa.optimization.optimize:No shadow prices were assigned to the network.


('ok', 'optimal')

In [9]:
nu.generators_t.status

name,coal,gas
snapshot,,
0,1.0,1.0
1,0.0,1.0
2,1.0,1.0
3,1.0,0.0


In [10]:
nu.objective

306150.0

In [11]:
nu.generators_t.p

name,coal,gas
snapshot,,
0,3900.0,100.0
1,0.0,800.0
2,4900.0,100.0
3,3000.0,0.0


## Minimum Down Time

Coal has a minimum down time, forcing it to go off longer than otherwise cost-optimal.

In [12]:
nu = pypsa.Network(snapshots=range(4))

nu.add("Bus", "bus")

nu.add(
    "Generator",
    "coal",
    bus="bus",
    committable=True,
    p_min_pu=0.3,
    marginal_cost=20,
    min_down_time=2,
    down_time_before=1,
    p_nom=10_000,
)

nu.add(
    "Generator",
    "gas",
    bus="bus",
    committable=True,
    marginal_cost=70,
    p_min_pu=0.1,
    p_nom=4_000,
)

nu.add("Load", "load", bus="bus", p_set=[3_000, 800, 3_000, 8_000])

In [13]:
nu.optimize()

Index(['bus'], dtype='object', name='name')


INFO:linopy.model: Solve problem using Gurobi solver


INFO:linopy.model:Solver options:
 - log_to_console: False


INFO:linopy.io: Writing time: 0.03s


Set parameter WLSAccessID


Set parameter WLSSecret


... (5 outputs left out)

Set parameter LogToConsole to value 0


INFO:linopy.constants: Optimization successful: 
Status: ok
Termination condition: optimal
Solution: 32 primals, 0 duals
Objective: 4.86e+05
Solver: gurobi
Runtime: 0.00s
MIP gap: 0.00e+00
Dual bound: 4.86e+05
Solver model: available
Solver message: 2



INFO:pypsa.optimization.optimize:No shadow prices were assigned to the network.


('ok', 'optimal')

In [14]:
nu.objective

486000.0

In [15]:
nu.generators_t.status

name,coal,gas
snapshot,,
0,0.0,1.0
1,0.0,1.0
2,1.0,0.0
3,1.0,0.0


In [16]:
nu.generators_t.p

name,coal,gas
snapshot,,
0,0.0,3000.0
1,0.0,800.0
2,3000.0,0.0
3,8000.0,0.0


## Start Up and Shut Down Costs

Now there are costs associated with shut down and start up events, which could incentivise longer up times of generators with high start-up and shut-down costs.

In [17]:
nu = pypsa.Network(snapshots=range(4))

nu.add("Bus", "bus")

nu.add(
    "Generator",
    "coal",
    bus="bus",
    committable=True,
    p_min_pu=0.3,
    marginal_cost=20,
    min_down_time=2,
    start_up_cost=5_000,
    p_nom=10_000,
)

nu.add(
    "Generator",
    "gas",
    bus="bus",
    committable=True,
    marginal_cost=70,
    p_min_pu=0.1,
    shut_down_cost=25,
    p_nom=4_000,
)

nu.add("Load", "load", bus="bus", p_set=[3_000, 800, 3_000, 8_000])

In [18]:
nu.optimize()

Index(['bus'], dtype='object', name='name')


INFO:linopy.model: Solve problem using Gurobi solver


INFO:linopy.model:Solver options:
 - log_to_console: False


INFO:linopy.io: Writing time: 0.03s


Set parameter WLSAccessID


Set parameter WLSSecret


... (5 outputs left out)

Set parameter LogToConsole to value 0


INFO:linopy.constants: Optimization successful: 
Status: ok
Termination condition: optimal
Solution: 32 primals, 0 duals
Objective: 4.91e+05
Solver: gurobi
Runtime: 0.00s
MIP gap: 0.00e+00
Dual bound: 4.91e+05
Solver model: available
Solver message: 2



INFO:pypsa.optimization.optimize:No shadow prices were assigned to the network.


('ok', 'optimal')

In [19]:
nu.objective

491025.0

In [20]:
nu.generators_t.status

name,coal,gas
snapshot,,
0,0.0,1.0
1,0.0,1.0
2,1.0,0.0
3,1.0,0.0


In [21]:
nu.generators_t.p

name,coal,gas
snapshot,,
0,0.0,3000.0
1,0.0,800.0
2,3000.0,0.0
3,8000.0,0.0


## Time-Varying Start Up and Shut Down Costs

Start-up and shut-down costs can also be provided as time series, varying by snapshot. This allows modeling scenarios where transition costs depend on time-of-day factors like fuel prices or market conditions.

In this example, a low-marginal-cost committable generator faces low start-up costs in snapshot 1 and low shut-down costs in snapshot 4, incentivizing it to start early and stay online late despite high stand-by costs.

In [22]:
nu = pypsa.Network(snapshots=range(5))

nu.add("Bus", "bus")

# Load peaks only in snapshot 2
load = pd.Series([0.0, 0.1, 10.0, 0.1, 0.1], index=nu.snapshots)

# Time-varying start-up costs: low in snapshot 1, high elsewhere
start_up_costs = pd.Series([0.0, 0.0, 100.0, 100.0, 100.0], index=nu.snapshots)

# Time-varying shut-down costs: low in snapshot 4, high elsewhere
shut_down_costs = pd.Series([100.0, 100.0, 100.0, 100.0, 0.0], index=nu.snapshots)

nu.add(
    "Generator",
    "cheap-committable",
    bus="bus",
    committable=True,
    p_nom=10,
    marginal_cost=0,
    stand_by_cost=10,
    p_min_pu=0.01,
    up_time_before=0,
    down_time_before=0,
    start_up_cost=start_up_costs,
    shut_down_cost=shut_down_costs,
)

# Fallback generator ensures feasibility
nu.add(
    "Generator",
    "expensive-fallback",
    bus="bus",
    committable=True,
    p_nom=10,
    marginal_cost=100,
    start_up_cost=0.1,
    shut_down_cost=0.1,
)

nu.add("Load", "load", bus="bus", p_set=load)

In [23]:
# View the time-varying costs
nu.generators_t.start_up_cost

name,cheap-committable
snapshot,
0,0.0
1,0.0
2,100.0
3,100.0
4,100.0


In [24]:
nu.generators.shut_down_cost

name
cheap-committable     0.0
expensive-fallback    0.1
Name: shut_down_cost, dtype: float64

In [25]:
nu.generators_t.shut_down_cost

name,cheap-committable
snapshot,
0,100.0
1,100.0
2,100.0
3,100.0
4,0.0


In [26]:
nu.optimize(log_to_console=False)

Index(['bus'], dtype='object', name='name')


INFO:linopy.model: Solve problem using Gurobi solver


INFO:linopy.model:Solver options:
 - log_to_console: False


INFO:linopy.io: Writing time: 0.03s


Set parameter WLSAccessID


Set parameter WLSSecret


... (5 outputs left out)

Set parameter LogToConsole to value 0


INFO:linopy.constants: Optimization successful: 
Status: ok
Termination condition: optimal
Solution: 40 primals, 0 duals
Objective: 4.00e+01
Solver: gurobi
Runtime: 0.00s
MIP gap: 0.00e+00
Dual bound: 4.00e+01
Solver model: available
Solver message: 2



INFO:pypsa.optimization.optimize:No shadow prices were assigned to the network.


('ok', 'optimal')

In [27]:
nu.generators_t.status

name,cheap-committable,expensive-fallback
snapshot,,
0,0.0,1.0
1,1.0,1.0
2,1.0,1.0
3,1.0,1.0
4,0.0,1.0


In [28]:
nu.generators_t.p

name,cheap-committable,expensive-fallback
snapshot,,
0,0.0,0.0
1,0.1,0.0
2,10.0,0.0
3,0.1,0.0
4,0.0,0.1


## Ramp Rate Limits

Ramp rate limits can be set for ramping up and down and are given as percentage of the nominal power that can be ramped up or down per snapshot. Note that the ramp limits apply per snapshot and are **not** weighted by the time step duration (`nu.snapshot_weightings`).

In [29]:
nu = pypsa.Network(snapshots=range(6))

nu.add("Bus", "bus")

nu.add(
    "Generator",
    "coal",
    bus="bus",
    marginal_cost=20,
    ramp_limit_up=0.1,
    ramp_limit_down=0.2,
    p_nom=10_000,
)

nu.add("Generator", "gas", bus="bus", marginal_cost=70, p_nom=4_000)

nu.add("Load", "load", bus="bus", p_set=[4_000, 7_000, 7_000, 7_000, 7_000, 3_000])

In [30]:
nu.optimize()

Index(['bus'], dtype='object', name='name')


INFO:linopy.model: Solve problem using Gurobi solver


INFO:linopy.model:Solver options:
 - log_to_console: False


INFO:linopy.io: Writing time: 0.02s


Set parameter WLSAccessID


Set parameter WLSSecret


... (5 outputs left out)

Set parameter LogToConsole to value 0


INFO:linopy.constants: Optimization successful: 
Status: ok
Termination condition: optimal
Solution: 12 primals, 54 duals
Objective: 9.50e+05
Solver: gurobi
Runtime: 0.00s
Dual bound: 9.50e+05
Solver model: available
Solver message: 2



INFO:pypsa.optimization.optimize:The shadow-prices of the constraints Generator-fix-p-lower, Generator-fix-p-upper, Generator-p-ramp_limit_up, Generator-p-ramp_limit_down were not assigned to the network.


('ok', 'optimal')

In [31]:
nu.generators_t.p

name,coal,gas
snapshot,,
0,4000.0,0.0
1,5000.0,2000.0
2,6000.0,1000.0
3,7000.0,0.0
4,5000.0,2000.0
5,3000.0,0.0


With capacity expansion (as long as unit is not committable):

In [32]:
nu = pypsa.Network(snapshots=range(6))

nu.add("Bus", "bus")

nu.add(
    "Generator",
    "coal",
    bus="bus",
    marginal_cost=20,
    ramp_limit_up=0.1,
    ramp_limit_down=0.2,
    p_nom_extendable=True,
    capital_cost=1e2,
)

nu.add("Generator", "gas", bus="bus", marginal_cost=70, p_nom=4000)

nu.add("Load", "load", bus="bus", p_set=[4000, 7000, 7000, 7000, 7000, 3000])

In [33]:
nu.optimize()

Index(['bus'], dtype='object', name='name')


INFO:linopy.model: Solve problem using Gurobi solver


INFO:linopy.model:Solver options:
 - log_to_console: False


INFO:linopy.io: Writing time: 0.02s


Set parameter WLSAccessID


Set parameter WLSSecret


... (5 outputs left out)

Set parameter LogToConsole to value 0


INFO:linopy.constants: Optimization successful: 
Status: ok
Termination condition: optimal
Solution: 13 primals, 55 duals
Objective: 1.68e+06
Solver: gurobi
Runtime: 0.00s
Dual bound: 1.68e+06
Solver model: available
Solver message: 2



INFO:pypsa.optimization.optimize:The shadow-prices of the constraints Generator-fix-p-lower, Generator-fix-p-upper, Generator-ext-p-lower, Generator-ext-p-upper, Generator-p-ramp_limit_up, Generator-p-ramp_limit_down were not assigned to the network.


('ok', 'optimal')

In [34]:
nu.generators.p_nom_opt

name
coal    5000.0
gas     4000.0
Name: p_nom_opt, dtype: float64

In [35]:
nu.generators_t.p

name,coal,gas
snapshot,,
0,4000.0,0.0
1,4500.0,2500.0
2,5000.0,2000.0
3,5000.0,2000.0
4,4000.0,3000.0
5,3000.0,0.0


Watch out for bad interactions, for example, when the ramp limit at start up or shut down is bigger than the regular ramp limit or minimum part load, which can lead to infeasibilities.

In [36]:
nu = pypsa.Network(snapshots=range(7))

nu.add("Bus", "bus")

# Can get bad interactions if SU > RU and p_min_pu; similarly if SD > RD
nu.add(
    "Generator",
    "coal",
    bus="bus",
    marginal_cost=20,
    committable=True,
    p_min_pu=0.05,
    initial_status=0,
    ramp_limit_start_up=0.1,
    ramp_limit_up=0.2,
    ramp_limit_down=0.25,
    ramp_limit_shut_down=0.15,
    p_nom=10_000,
)

nu.add("Generator", "gas", bus="bus", marginal_cost=70, p_nom=10_000)

nu.add("Load", "load", bus="bus", p_set=[0, 200, 7_000, 7_000, 7_000, 2_000, 0])

In [37]:
nu.optimize()

Index(['bus'], dtype='object', name='name')


INFO:linopy.model: Solve problem using Gurobi solver


INFO:linopy.model:Solver options:
 - log_to_console: False


INFO:linopy.io: Writing time: 0.04s


Set parameter WLSAccessID


Set parameter WLSSecret


... (5 outputs left out)

Set parameter LogToConsole to value 0


INFO:linopy.constants: Optimization successful: 
Status: ok
Termination condition: optimal
Solution: 35 primals, 0 duals
Objective: 1.15e+06
Solver: gurobi
Runtime: 0.00s
MIP gap: 0.00e+00
Dual bound: 1.15e+06
Solver model: available
Solver message: 2



INFO:pypsa.optimization.optimize:No shadow prices were assigned to the network.


('ok', 'optimal')

In [38]:
nu.generators_t.p

name,coal,gas
snapshot,,
0,0.0,0.0
1,0.0,200.0
2,1000.0,6000.0
3,3000.0,4000.0
4,4000.0,3000.0
5,1500.0,500.0
6,0.0,0.0


In [39]:
nu.generators_t.status

name,coal,gas
snapshot,,
0,0.0,0.0
1,0.0,0.0
2,1.0,0.0
3,1.0,0.0
4,1.0,0.0
5,1.0,0.0
6,0.0,0.0


## Load Shedding in Blocks

Unit commitment also applies to demand. Here, load shedding is a committable generator with a high marginal cost. With `p_min_pu=0.2`, it sheds either nothing or at least 20 MW.

In the second snapshot, only 5 MW are missing. A full block of 20 MW is shed and coal reduces its output.

In [40]:
nu = pypsa.Network(snapshots=range(4))

nu.add("Bus", "bus")

nu.add("Generator", "coal", bus="bus", marginal_cost=20, p_nom=100)

nu.add(
    "Generator",
    "load shedding",
    bus="bus",
    committable=True,
    p_min_pu=0.2,
    marginal_cost=3_000,
    p_nom=100,
)

nu.add("Load", "load", bus="bus", p_set=[90, 105, 120, 100])

In [41]:
nu.optimize()

Index(['bus'], dtype='object', name='name')


INFO:linopy.model: Solve problem using Gurobi solver


INFO:linopy.model:Solver options:
 - log_to_console: False


INFO:linopy.io: Writing time: 0.04s


Set parameter WLSAccessID


Set parameter WLSSecret


... (5 outputs left out)

Set parameter LogToConsole to value 0


INFO:linopy.constants: Optimization successful: 
Status: ok
Termination condition: optimal
Solution: 20 primals, 0 duals
Objective: 1.28e+05
Solver: gurobi
Runtime: 0.00s
MIP gap: 0.00e+00
Dual bound: 1.28e+05
Solver model: available
Solver message: 2



INFO:pypsa.optimization.optimize:No shadow prices were assigned to the network.


('ok', 'optimal')

In [42]:
nu.generators_t.p

name,coal,load shedding
snapshot,,
0,90.0,0.0
1,85.0,20.0
2,100.0,20.0
3,100.0,0.0


## Flexible Demand with Start-Up Cost

A flexible demand is worth 50 EUR/MWh, takes between 20 and 40 MW when it runs, and costs 300 EUR to switch on. It is a committable generator with `sign=-1`, so that its dispatch is consumption. The negative marginal cost is its willingness to pay.

The demand runs while coal has spare capacity. It switches off in the third snapshot, where it would need gas at 70 EUR/MWh.

In [43]:
nu = pypsa.Network(snapshots=range(4))

nu.add("Bus", "bus")

nu.add("Generator", "coal", bus="bus", marginal_cost=20, p_nom=100)

nu.add("Generator", "gas", bus="bus", marginal_cost=70, p_nom=100)

nu.add(
    "Generator",
    "flexible demand",
    bus="bus",
    sign=-1,
    committable=True,
    p_min_pu=0.5,
    marginal_cost=-50,
    start_up_cost=300,
    up_time_before=0,
    p_nom=40,
)

nu.add("Load", "load", bus="bus", p_set=[60, 70, 110, 60])

In [44]:
nu.optimize()

Index(['bus'], dtype='object', name='name')


INFO:linopy.model: Solve problem using Gurobi solver


INFO:linopy.model:Solver options:
 - log_to_console: False


INFO:linopy.io: Writing time: 0.03s


Set parameter WLSAccessID


Set parameter WLSSecret


... (5 outputs left out)

Set parameter LogToConsole to value 0


INFO:linopy.constants: Optimization successful: 
Status: ok
Termination condition: optimal
Solution: 24 primals, 0 duals
Objective: 3.80e+03
Solver: gurobi
Runtime: 0.00s
MIP gap: 0.00e+00
Dual bound: 3.80e+03
Solver model: available
Solver message: 2



INFO:pypsa.optimization.optimize:No shadow prices were assigned to the network.


('ok', 'optimal')

In [45]:
nu.generators_t.p

name,coal,gas,flexible demand
snapshot,,,
0,100.0,0.0,40.0
1,100.0,0.0,30.0
2,100.0,10.0,0.0
3,100.0,0.0,40.0


## Rolling Horizon

The unit commitment optimisation can be combined with a rolling horizon optimisation, i.e. solving the snapshots sequentially in batches. This can be done manually (as shown here) or automatically, using `nu.optimize.optimize_with_rolling_horizon()`.

In [46]:
sets_of_snapshots = 6
p_set = [4_000, 5_000, 700, 800, 4_000]

nu = pypsa.Network(snapshots=range(len(p_set) * sets_of_snapshots))

nu.add("Bus", "bus")

nu.add(
    "Generator",
    "coal",
    bus="bus",
    committable=True,
    p_min_pu=0.3,
    marginal_cost=20,
    min_down_time=2,
    min_up_time=3,
    up_time_before=1,
    ramp_limit_up=1,
    ramp_limit_down=1,
    ramp_limit_start_up=1,
    ramp_limit_shut_down=1,
    shut_down_cost=150,
    start_up_cost=200,
    p_nom=10_000,
)

nu.add(
    "Generator",
    "gas",
    bus="bus",
    committable=True,
    marginal_cost=70,
    p_min_pu=0.1,
    up_time_before=2,
    min_up_time=3,
    shut_down_cost=20,
    start_up_cost=50,
    p_nom=1_000,
)

nu.add("Load", "load", bus="bus", p_set=p_set * sets_of_snapshots)

In [47]:
overlap = 2
for i in range(sets_of_snapshots):
    snapshots = nu.snapshots[i * len(p_set) : (i + 1) * len(p_set) + overlap]
    nu.optimize(snapshots=snapshots)

Index(['bus'], dtype='object', name='name')


INFO:linopy.model: Solve problem using Gurobi solver


INFO:linopy.model:Solver options:
 - log_to_console: False


INFO:linopy.io: Writing time: 0.04s


Set parameter WLSAccessID


Set parameter WLSSecret


... (74 outputs left out)

obj: 100 rows, 40 columns, 211 nonzeros


Set parameter LogToConsole to value 0


INFO:linopy.constants: Optimization successful: 
Status: ok
Termination condition: optimal
Solution: 40 primals, 0 duals
Objective: 3.70e+05
Solver: gurobi
Runtime: 0.00s
MIP gap: 0.00e+00
Dual bound: 3.70e+05
Solver model: available
Solver message: 2



INFO:pypsa.optimization.optimize:No shadow prices were assigned to the network.


In [48]:
pd.concat(
    {"Active": nu.generators_t.status.astype(bool), "Output": nu.generators_t.p}, axis=1
)

Active         Output       
name       coal    gas    coal    gas
snapshot                             
0          True   True  3900.0  100.0
1          True   True  4900.0  100.0
2         False   True     0.0  700.0
3         False   True     0.0  800.0
4          True  False  4000.0    0.0
5          True  False  4000.0    0.0
6          True  False  5000.0    0.0
7         False   True     0.0  700.0
8         False   True     0.0  800.0
9          True   True  3900.0  100.0
10         True  False  4000.0    0.0
11         True  False  5000.0    0.0
12        False   True     0.0  700.0
13        False   True     0.0  800.0
14         True   True  3900.0  100.0
15         True  False  4000.0    0.0
16         True  False  5000.0    0.0
17        False   True     0.0  700.0
18        False   True     0.0  800.0
19         True   True  3900.0  100.0
20         True  False  4000.0    0.0
21         True  False  5000.0    0.0
22        False   True     0.0  700.0
23        False   True     0.0  800.0
24         True   True  3900.0  100.0
25         True  False  4000.0    0.0
26         True  False  5000.0    0.0
27        False   True     0.0  700.0
28        False   True     0.0  800.0
29         True   True  3900.0  100.0